# Study 2 — Linear KAN-FNO: Training & Symbolic Recovery

**What this notebook does:**
1. Installs the `ls-kan-fno` package from GitHub
2. Loads `dataset_v3.h5` from your Google Drive
3. Verifies the pipeline at exact initialization (field_loss ≈ 0)
4. Trains `KANTauTheta` (3 parameters) for 100 epochs
5. Plots training curves
6. Runs symbolic recovery — confirms trained `ctrl ≈ [1, −1, 1]` (= exact x²)

**Expected outcome:** The 3 learned control points converge to `[1, −1, 1]` to
3–4 decimal places, confirming that gradient descent can recover the correct
symbolic form of the double-contraction operator from FFT-generated data.

**Runtime estimate:** ~15–25 min on a T4 GPU (100 epochs × 50 batches with B=32).

---
**Before running:** Go to `Runtime → Change runtime type → T4 GPU`.

## §1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## §2 — Clone repository and install package

Replace `YOUR_USERNAME/YOUR_REPO` with your GitHub repository path.
The clone targets `/content/ls-kan-fno`; `LS_KAN_FNO/` is the installable
sub-directory containing `pyproject.toml`.

In [ ]:
GITHUB_REPO = "YOUR_USERNAME/YOUR_REPO"   # ← EDIT THIS

import subprocess, sys

# Clone (skip if already present — useful after a Colab session restart)
result = subprocess.run(
    ["git", "clone", f"https://github.com/{GITHUB_REPO}.git", "/content/ls-kan-fno"],
    capture_output=True, text=True
)
if result.returncode == 0:
    print("Cloned successfully.")
elif "already exists" in result.stderr:
    print("Repo already present — pulling latest changes.")
    subprocess.run(["git", "-C", "/content/ls-kan-fno", "pull"], check=True)
else:
    print(result.stderr)
    raise RuntimeError("git clone failed")

# Install the package in editable mode (quiet — suppress pip noise)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e",
     "/content/ls-kan-fno/LS_KAN_FNO", "-q"],
    check=True
)
print("Package installed.")

# Add to sys.path and set working directory
import os, sys
REPO_ROOT = "/content/ls-kan-fno/LS_KAN_FNO"
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
os.chdir(REPO_ROOT)
print(f"Working directory: {os.getcwd()}")

## §3 — Configure paths

Set `DATA_PATH` to where you placed `dataset_v3.h5` on your Drive,
and `RUN_DIR` to where checkpoints and logs should be saved.
**Edit this cell before running anything else.**

In [ ]:
# ── EDIT THESE TWO LINES ────────────────────────────────────────────────────
DATA_PATH = "/content/drive/MyDrive/ls_kan_fno/data/dataset_v3.h5"
RUN_DIR   = "/content/drive/MyDrive/ls_kan_fno/runs/linear_study2"
# ────────────────────────────────────────────────────────────────────────────

from pathlib import Path
Path(RUN_DIR).mkdir(parents=True, exist_ok=True)

data_ok = Path(DATA_PATH).exists()
print(f"DATA_PATH : {DATA_PATH}  {'✓ found' if data_ok else '✗ NOT FOUND — see §4'}")
print(f"RUN_DIR   : {RUN_DIR}  (created)")

## §4 — (Optional) Generate dataset_v3 on this machine

**Skip this cell if you already have `dataset_v3.h5` on Drive.**

If `DATA_PATH` was not found above, run this cell to regenerate the dataset
on Colab's CPU (~5 min) and save it directly to Drive.

In [ ]:
from pathlib import Path

if Path(DATA_PATH).exists():
    print(f"dataset_v3.h5 already exists at {DATA_PATH} — skipping generation.")
else:
    print(f"Generating dataset_v3.h5 → {DATA_PATH}")
    print("This takes ~5 min on CPU. Grab a coffee.")

    from generation.generate_dataset import generate
    from utils.config_loader import load_config

    cfg = load_config("configs/data.yaml")
    cfg["tag"]        = "v3"
    cfg["output_dir"] = str(Path(DATA_PATH).parent)
    generate(cfg)

    assert Path(DATA_PATH).exists(), "Generation finished but file not found — check output_dir."
    print(f"\ndataset_v3.h5 saved to {DATA_PATH}")

## §5 — GPU check

In [ ]:
import torch

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU  : {props.name}")
    print(f"VRAM : {props.total_memory / 1e9:.1f} GB")
    print(f"Torch: {torch.__version__}")
else:
    print("WARNING: No GPU detected.")
    print("Go to Runtime → Change runtime type → T4 GPU, then re-run from §1.")
    print("Training on CPU will work but takes ~10× longer.")

## §6 — Sanity check: field_loss at exact initialization

With `ctrl = [1, −1, 1]` (the exact x² representation), the LSFNO with K=140
is identical to running 140 fixed-point iterations at α₀=10.0. Since dataset_v3
was generated with the same α₀ and tol=1e-7, the model's output should match
the stored `eps_star` to within machine precision.

**Expected:** `field_loss < 0.001` (we've measured ~0.0001% on this dataset).
If this fails, something is wrong with the data path or the package installation.

In [ ]:
import torch
from utils.config_loader import load_config
from models.kan_tau_theta import KANTauTheta
from models.ls_fno import LSFNO
from datasets.micromechanics import DataLoaderFactory
from training.losses import field_loss

# Load config and patch paths
cfg = load_config("configs/training_colab.yaml")
cfg["train_path"] = DATA_PATH
cfg["val_path"]   = DATA_PATH
cfg["test_path"]  = DATA_PATH

# Build model with FROZEN exact control points
tau = KANTauTheta(R=1.0, shared=True, trainable=False, n_comp=3)
model = LSFNO.from_config(cfg, tau_theta=tau)
model.eval()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model  = model.to(device)

loaders = DataLoaderFactory.from_config(cfg)
batch   = next(iter(loaders["val"]))
batch   = {k: v.to(device) if torch.is_tensor(v) else v for k, v in batch.items()}

import time
with torch.no_grad():
    t0 = time.time()
    eps_pred = model(batch["C_field"], batch["eps_bar"], use_checkpointing=False)
    dt = time.time() - t0
    fl = field_loss(eps_pred, batch["eps_star"]).item()

print(f"field_loss at exact ctrl=[1,−1,1]: {fl:.6f}  ({fl*100:.4f}%)")
print(f"Forward pass time (B={batch['C_field'].shape[0]}, K=140): {dt:.2f}s")

assert fl < 0.001, f"Sanity check FAILED: field_loss={fl:.4f}. Check DATA_PATH and package install."
print("\nSanity check PASSED ✓  — pipeline is correctly wired.")

## §7 — Train KANTauTheta (100 epochs)

The only trainable parameters are the 3 B-spline control points `ctrl`.
Everything else (the Green operator, FFT steps, Voigt↔Mandel conversions)
is fixed analytic computation.

Epoch log columns:
- `train_field` / `val_field` — relative L2 strain field error
- `train_eff`   / `val_eff`   — relative directional-modulus error
- `gamma_theta_max`           — empirical Lipschitz constant of τ_θ
- `gamma_theta_bound`         — theoretical limit 2/(κ+1) ≈ 0.182

Checkpoints are saved every 10 epochs and whenever validation loss improves.

In [ ]:
from utils.config_loader import load_config
from training.trainer import Trainer

cfg = load_config("configs/training_colab.yaml")
cfg["train_path"] = DATA_PATH
cfg["val_path"]   = DATA_PATH
cfg["test_path"]  = DATA_PATH
cfg["output_dir"] = RUN_DIR

trainer = Trainer(cfg)
history = trainer.fit()

print(f"\nTraining complete.")
print(f"Best val loss : {trainer.best_val_loss:.6f}")
print(f"Checkpoints   : {RUN_DIR}/")

## §8 — Training curves

Two panels:
- **Left**: field loss (log scale) — should decrease monotonically and plateau near the machine-precision floor (~0.0001%)
- **Right**: γ_θ contractivity check — must stay below the dashed red line 2/(κ+1) ≈ 0.182 throughout training

In [ ]:
import json
import matplotlib.pyplot as plt
from pathlib import Path

history_path = Path(RUN_DIR) / "history.jsonl"
rows = [json.loads(line) for line in history_path.read_text().strip().splitlines()]

epochs      = [r["epoch"]       for r in rows]
train_field = [r["train_field"] for r in rows]
val_rows    = [r for r in rows if "val_field" in r]
gamma_rows  = [r for r in rows if "gamma_theta_max" in r]

kappa = 10.0   # from experiment.yaml
gamma_bound = 2.0 / (kappa + 1.0)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# ── Left: field loss ──────────────────────────────────────────────────────
ax = axes[0]
ax.semilogy(epochs, train_field, color="steelblue", lw=1.5, label="train field loss")
if val_rows:
    ax.semilogy([r["epoch"] for r in val_rows],
                [r["val_field"] for r in val_rows],
                color="coral", lw=1.5, ls="--", label="val field loss")
ax.axhline(1e-3, color="gray", lw=0.8, ls=":", label="0.1% threshold")
ax.set_xlabel("Epoch")
ax.set_ylabel("Relative L2 field error")
ax.set_title("Field loss (log scale)")
ax.legend()
ax.grid(True, which="both", alpha=0.3)

# ── Right: contractivity ──────────────────────────────────────────────────
ax = axes[1]
if gamma_rows:
    ax.plot([r["epoch"] for r in gamma_rows],
            [r["gamma_theta_max"] for r in gamma_rows],
            color="steelblue", lw=1.5, label="γ_θ max (empirical)")
    ax.plot([r["epoch"] for r in gamma_rows],
            [r["gamma_theta_p99"] for r in gamma_rows],
            color="steelblue", lw=1, ls="--", alpha=0.6, label="γ_θ p99")
ax.axhline(gamma_bound, color="red", lw=1.5, ls="--",
           label=f"Bound 2/(κ+1) = {gamma_bound:.3f}")
ax.set_xlabel("Epoch")
ax.set_ylabel("γ_θ (Lipschitz constant)")
ax.set_title("Contractivity check")
ax.legend()
ax.grid(True, alpha=0.3)

plt.suptitle("Study 2 — Linear KAN-FNO training", fontsize=12)
plt.tight_layout()
plot_path = Path(RUN_DIR) / "training_curves.png"
plt.savefig(plot_path, dpi=150, bbox_inches="tight")
print(f"Plot saved to {plot_path}")
plt.show()

## §9 — Inspect trained control points

Quick look at what `ctrl` converged to before running the full symbolic recovery.
Expected: `[1.xxx, −1.xxx, 1.xxx]` close to the exact `[1, −1, 1]`.

In [ ]:
import torch
from pathlib import Path

ckpt_path = Path(RUN_DIR) / "best_checkpoint.pt"
ckpt      = torch.load(ckpt_path, map_location="cpu", weights_only=False)

ctrl = ckpt["model_state_dict"]["tau_theta.ctrl"].numpy()
print(f"Epoch of best checkpoint : {ckpt['epoch']}")
print(f"Val loss at best         : {ckpt['val_loss']:.6f}")
print()
print(f"Trained ctrl : [{ctrl[0]:+.6f}, {ctrl[1]:+.6f}, {ctrl[2]:+.6f}]")
print(f"Exact   ctrl : [ 1.000000, -1.000000,  1.000000]")
print(f"Max deviation: {max(abs(ctrl[0]-1), abs(ctrl[1]+1), abs(ctrl[2]-1)):.2e}")

## §10 — Symbolic recovery

Full pipeline:
1. Reconstruct the learned edge function φ(x) over x ∈ [−1, 1]
2. Compare to x² pointwise
3. Fit to a candidate library: x², x, |x|, const, x³, x⁴
4. Report R² for each candidate and give a PASS/FAIL verdict

**PASS criteria:**
- `max |ctrl − [1,−1,1]| < 0.01`
- `R²(x²) ≥ 0.9999`
- `x²` is the best-fitting candidate

In [ ]:
import sys
from pathlib import Path

# Make sure symbolic/ is importable
if "/content/ls-kan-fno/LS_KAN_FNO" not in sys.path:
    sys.path.insert(0, "/content/ls-kan-fno/LS_KAN_FNO")

from symbolic.recover import recover

ckpt_path = str(Path(RUN_DIR) / "best_checkpoint.pt")
results   = recover(ckpt_path, plot=True, n_grid=300)

if results["passed"]:
    print("Study 2 COMPLETE ✓")
    print(f"  ctrl converged to {results['ctrl'].round(4).tolist()}")
    print(f"  R²(x²) = {results['fits']['x²']['r2']:.6f}")
else:
    print("Study 2 not yet complete — check the training curves and run more epochs if needed.")

## §11 — (Optional) LBFGS refinement

With only 3 parameters, Adam is usually sufficient. If `ctrl` hasn't converged
tightly to `[1, −1, 1]` after 100 epochs, a handful of LBFGS steps on a small
batch can close the gap.

**Only run this if the symbolic recovery in §10 did not PASS.**

In [ ]:
# Uncomment to run LBFGS refinement

# from utils.config_loader import load_config
# from training.trainer import Trainer
#
# cfg = load_config("configs/training_colab.yaml")
# cfg["train_path"]  = DATA_PATH
# cfg["val_path"]    = DATA_PATH
# cfg["test_path"]   = DATA_PATH
# cfg["output_dir"]  = RUN_DIR
# cfg["lbfgs_steps"] = 50        # more steps for tighter convergence
#
# # Reload trainer from best checkpoint and run LBFGS
# trainer = Trainer(cfg)
# final_loss = trainer.fit_lbfgs()
# print(f"LBFGS final loss: {final_loss:.6e}")
#
# # Re-run symbolic recovery
# from symbolic.recover import recover
# results = recover(str(Path(RUN_DIR) / "last_lbfgs_checkpoint.pt"), plot=True)

---
## Summary

| Quantity | Target | Measured |
|---|---|---|
| ctrl[0] | 1.0000 | *(fill after training)* |
| ctrl[1] | −1.0000 | *(fill after training)* |
| ctrl[2] | 1.0000 | *(fill after training)* |
| max δ_ctrl | < 0.01 | *(fill after recovery)* |
| R²(x²) | ≥ 0.9999 | *(fill after recovery)* |
| Best symbolic fit | x² | *(fill after recovery)* |
| val field loss (best) | < 0.001 | *(fill after training)* |
| γ_θ max | < 0.182 | *(fill after training)* |

**Next notebook:** `02_symbolic_recovery.ipynb` — deeper analysis of the trained
edges, per-component breakdown, and comparison against the Yarotsky-MLP baseline.